In [5]:
"""
=============================================================================
NFHS-5 District-Level Maternal Health Analysis — Starter Script
=============================================================================
Author: Kaushal Kumar
Project: District-Level Disparities in Maternal Healthcare Access Across India

This script:
  1. Downloads NFHS-5 district factsheet data from GitHub
  2. Selects maternal health indicators
  3. Cleans and prepares the dataset
  4. Produces descriptive statistics
  5. Creates initial choropleth maps

Requirements:
  pip install pandas numpy matplotlib geopandas requests openpyxl
=============================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# STEP 1: LOAD NFHS-5 DISTRICT FACTSHEET DATA
# ============================================================================

# Option A: Load from Pratap Vardhan's GitHub (pre-cleaned CSVs)
# URL: https://github.com/pratapvardhan/NFHS-5
# Download the district-level CSV manually or use:

NFHS5_DISTRICT_URL = (
     "https://raw.githubusercontent.com/pratapvardhan/NFHS-5/"
    "master/NFHS-5-Districts.csv"
)

print("=" * 70)
print("STEP 1: Loading NFHS-5 District Factsheet Data")
print("=" * 70)

try:
    df_raw = pd.read_csv(NFHS5_DISTRICT_URL)
    print(f"Loaded from GitHub: {df_raw.shape[0]} districts, {df_raw.shape[1]} indicators")
except Exception as e:
    print(f"GitHub download failed: {e}")
    print("\nFALLBACK: Download manually from:")
    print("  1. https://github.com/pratapvardhan/NFHS-5")
    print("  2. https://www.data.gov.in/catalog/national-family-health-survey-5-nfhs-5-india-districts-factsheet-data")
    print("\nThen load with:")
    print("  df_raw = pd.read_csv('path/to/your/downloaded/file.csv')")
    # Create a placeholder so the rest of the script structure is visible
    df_raw = pd.DataFrame()

# Quick look at the data
if not df_raw.empty:
    print(f"\nColumns ({len(df_raw.columns)}):")
    for i, col in enumerate(df_raw.columns):
        print(f"  {i:3d}. {col}")




STEP 1: Loading NFHS-5 District Factsheet Data
Loaded from GitHub: 35464 districts, 8 indicators

Columns (8):
    0. State
    1. State-Code
    2. District
    3. Indicator
    4. NFHS-5
    5. NFHS-4
    6. NFHS-5-note
    7. NFHS-4-note


In [6]:
df_raw.head()

,State,State-Code,District,Indicator,NFHS-5,NFHS-4,NFHS-5-note,NFHS-4-note
0,Andaman and Nicobar Island,AN,Nicobar,1. Female population age 6 years and above who...,78.0,77.2,NaN,NaN
1,Andaman and Nicobar Island,AN,Nicobar,2. Population below age 15 years (%),23.0,25.4,NaN,NaN
2,Andaman and Nicobar Island,AN,Nicobar,3. Sex ratio of the total population (females ...,973.0,957.0,NaN,NaN
3,Andaman and Nicobar Island,AN,Nicobar,4. Sex ratio at birth for children born in the...,927.0,1060.0,NaN,NaN
4,Andaman and Nicobar Island,AN,Nicobar,5. Children under age 5 years whose birth was ...,98.0,99.3,NaN,NaN


In [9]:
import pandas as pd

# Extract indicator number from the text (e.g., "42. Institutional births (%)" → 42)
df_raw['ind_num'] = df_raw['Indicator'].str.extract(r'^(\d+)\.').astype(int)

# Pivot: each district becomes one row, each indicator becomes a column
df_wide = df_raw.pivot_table(
    index=['State', 'State-Code', 'District'],
    columns='ind_num',
    values='NFHS-5',
    aggfunc='first'
).reset_index()

# Rename columns to meaningful names
col_names = {
    1: 'female_school_attendance',
    14: 'female_literacy',
    15: 'women_10yr_schooling',
    16: 'child_marriage',
    7: 'hh_electricity',
    8: 'improved_water',
    9: 'improved_sanitation',
    10: 'clean_fuel',
    12: 'health_insurance',
    32: 'anc_first_trimester',
    33: 'anc_4plus_visits',
    35: 'ifa_100days',
    38: 'postnatal_care_2days',
    39: 'oop_expenditure_delivery',
    42: 'institutional_births',
    43: 'institutional_births_public',
    45: 'skilled_birth_attendance',
    46: 'csection_rate',
    73: 'child_stunting',
    76: 'child_underweight',
    84: 'women_anaemic',
}

df_wide = df_wide.rename(columns=col_names)

# Keep only the columns we need
keep = ['State', 'State-Code', 'District'] + list(col_names.values())
df_final = df_wide[[c for c in keep if c in df_wide.columns]]

print(f"Shape: {df_final.shape}")
print(df_final.head())
df_final.to_csv('nfhs5_maternal_health_wide.csv', index=False)

Shape: (341, 24)
ind_num                       State State-Code                District  \
0        Andaman and Nicobar Island         AN                 Nicobar   
1        Andaman and Nicobar Island         AN  North & Middle Andaman   
2        Andaman and Nicobar Island         AN           South Andaman   
3                    Andhra Pradesh         AP               Anantapur   
4                    Andhra Pradesh         AP                Chittoor   

ind_num  female_school_attendance  female_literacy  women_10yr_schooling  \
0                            78.0             87.5                  53.5   
1                            82.7             84.0                  41.0   
2                            84.7             86.7                  57.5   
3                            59.5             63.6                  31.3   
4                            65.6             69.3                  40.3   

ind_num  child_marriage  hh_electricity  improved_water  improved_sanitation  \
0

In [10]:
df_final.head()

ind_num,State,State-Code,District,female_school_attendance,female_literacy,women_10yr_schooling,child_marriage,hh_electricity,improved_water,improved_sanitation,...,ifa_100days,postnatal_care_2days,oop_expenditure_delivery,institutional_births,institutional_births_public,skilled_birth_attendance,csection_rate,child_stunting,child_underweight,women_anaemic
0,Andaman and Nicobar Island,AN,Nicobar,78.0,87.5,53.5,11.4,97.9,98.8,83.5,...,72.6,85.1,2278.0,97.8,96.7,98.6,11.5,21.6,24.6,38.3
1,Andaman and Nicobar Island,AN,North & Middle Andaman,82.7,84.0,41.0,15.4,93.2,92.2,86.4,...,83.7,92.5,1904.0,97.7,95.0,98.3,12.9,27.0,42.8,62.1
2,Andaman and Nicobar Island,AN,South Andaman,84.7,86.7,57.5,17.1,99.6,97.9,89.3,...,81.0,88.1,3460.0,99.5,83.8,96.9,37.1,21.1,17.4,57.7
3,Andhra Pradesh,AP,Anantapur,59.5,63.6,31.3,37.3,99.6,98.8,71.3,...,72.0,89.1,2682.0,94.7,60.3,98.3,22.2,36.0,40.6,50.5
4,Andhra Pradesh,AP,Chittoor,65.6,69.3,40.3,28.1,99.7,98.5,74.6,...,71.7,93.8,2533.0,97.1,66.5,97.6,26.4,27.1,27.9,51.8


In [11]:
# ============================================================================
# STEP 2: IDENTIFY AND SELECT MATERNAL HEALTH COLUMNS
# ============================================================================

print("\n" + "=" * 70)
print("STEP 2: Selecting Maternal Health Indicators")
print("=" * 70)

# Common maternal health indicator keywords in NFHS-5 factsheets
MATERNAL_KEYWORDS = [
    'antenatal', 'ANC', 'anc',
    'institutional births', 'institutional deliveries',
    'skilled health personnel', 'skilled birth',
    'postnatal', 'post-natal', 'PNC',
    'caesarean', 'c-section', 'C-section',
    'delivery', 'births',
    'maternal',
    'iron', 'IFA',         # iron-folic acid supplementation
    'neonatal', 'neo-natal',
]

# Auto-detect maternal health columns
def find_maternal_columns(df, keywords):
    """Find columns whose names contain any of the maternal health keywords."""
    maternal_cols = []
    for col in df.columns:
        col_lower = col.lower()
        if any(kw.lower() in col_lower for kw in keywords):
            maternal_cols.append(col)
    return maternal_cols

if not df_final.empty:
    maternal_cols = find_maternal_columns(df_final, MATERNAL_KEYWORDS)
    print(f"\nFound {len(maternal_cols)} maternal health columns:")
    for col in maternal_cols:
        print(f"  - {col}")

    # Also keep identifier columns
    id_keywords = ['state', 'district', 'State', 'District', 'area', 'Area']
    id_cols = [c for c in df_final.columns if any(k in c for k in id_keywords)]
    print(f"\nIdentifier columns: {id_cols}")



STEP 2: Selecting Maternal Health Indicators

Found 10 maternal health columns:
  - female_school_attendance
  - health_insurance
  - anc_first_trimester
  - anc_4plus_visits
  - ifa_100days
  - postnatal_care_2days
  - oop_expenditure_delivery
  - institutional_births
  - institutional_births_public
  - skilled_birth_attendance

Identifier columns: ['State', 'State-Code', 'District']


In [12]:
# ============================================================================
# STEP 3: BUILD ANALYSIS DATASET
# ============================================================================

print("\n" + "=" * 70)
print("STEP 3: Building Clean Analysis Dataset")
print("=" * 70)

if not df_final.empty and maternal_cols:
    # Select relevant columns
    keep_cols = id_cols + maternal_cols
    df = df_final[keep_cols].copy()

    # Convert indicator columns to numeric (they may be strings with % signs)
    for col in maternal_cols:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace('%', '').str.strip(),
            errors='coerce'
        )

    # Drop rows where all maternal indicators are NaN
    df = df.dropna(subset=maternal_cols, how='all')

    print(f"Analysis dataset: {df.shape[0]} districts, {df.shape[1]} columns")
    print(f"Missing values per column:")
    print(df[maternal_cols].isnull().sum().to_string())

    # Save cleaned dataset
    df.to_csv('nfhs5_maternal_health_clean.csv', index=False)
    print("\nSaved: nfhs5_maternal_health_clean.csv")




STEP 3: Building Clean Analysis Dataset
Analysis dataset: 341 districts, 13 columns
Missing values per column:
ind_num
female_school_attendance       0
health_insurance               0
anc_first_trimester            0
anc_4plus_visits               0
ifa_100days                    0
postnatal_care_2days           0
oop_expenditure_delivery       1
institutional_births           0
institutional_births_public    0
skilled_birth_attendance       0

Saved: nfhs5_maternal_health_clean.csv


In [13]:
# ============================================================================
# STEP 4: DESCRIPTIVE STATISTICS
# ============================================================================

print("\n" + "=" * 70)
print("STEP 4: Descriptive Statistics")
print("=" * 70)

if not df_final.empty and maternal_cols:
    # Summary statistics
    desc_stats = df_final[maternal_cols].describe().round(2)
    print("\n--- Summary Statistics ---")
    print(desc_stats.to_string())

    # State-level aggregation (mean of district values)
    state_col = [c for c in id_cols if 'state' in c.lower() or 'State' in c]
    if state_col:
        state_col = state_col[0]
        state_means = df_final.groupby(state_col)[maternal_cols].mean().round(2)
        print(f"\n--- State-wise Means (first 3 indicators) ---")
        print(state_means.iloc[:, :3].to_string())

        # Save state-level summary
        state_means.to_csv('nfhs5_maternal_state_means.csv')
        print("\nSaved: nfhs5_maternal_state_means.csv")

    # Identify worst-performing districts
    # Pick the first maternal indicator as example (usually institutional delivery)
    example_indicator = maternal_cols[0]
    print(f"\n--- Bottom 20 Districts by: {example_indicator} ---")
    bottom_20 = df_final.nsmallest(20, example_indicator)[id_cols + [example_indicator]]
    print(bottom_20.to_string(index=False))

    print(f"\n--- Top 20 Districts by: {example_indicator} ---")
    top_20 = df_final.nlargest(20, example_indicator)[id_cols + [example_indicator]]
    print(top_20.to_string(index=False))





STEP 4: Descriptive Statistics

--- Summary Statistics ---
ind_num  female_school_attendance  health_insurance  anc_first_trimester  anc_4plus_visits  ifa_100days  postnatal_care_2days  oop_expenditure_delivery  institutional_births  institutional_births_public  skilled_birth_attendance
count                      341.00            341.00               341.00            341.00       341.00                341.00                    340.00                341.00                       341.00                    341.00
mean                        73.45             36.74                72.23             62.72        47.64                 76.93                   4213.95                 88.28                        61.47                     88.98
std                         11.27             20.75                14.86             21.46        20.75                 16.54                   2590.24                 14.28                        17.82                     11.67
min                     

In [14]:
top_20

ind_num,State,State-Code,District,female_school_attendance
193,Kerala,KL,Kottayam,97.9
187,Kerala,KL,Alappuzha,97.8
199,Kerala,KL,Thrissur,97.8
259,Mizoram,MZ,Aizawl,97.8
194,Kerala,KL,Kozhikode,97.7
197,Kerala,KL,Pathanamthitta,97.7
188,Kerala,KL,Ernakulam,97.2
266,Mizoram,MZ,Serchhip,96.8
190,Kerala,KL,Kannur,96.7
192,Kerala,KL,Kollam,96.6


In [15]:
# ============================================================================
# STEP 5: INITIAL VISUALIZATIONS
# ============================================================================

print("\n" + "=" * 70)
print("STEP 5: Visualizations")
print("=" * 70)

if not df_final.empty and maternal_cols:

    # --- 5a: Distribution of key indicators ---
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Distribution of Maternal Health Indicators Across Indian Districts (NFHS-5)',
                 fontsize=14, fontweight='bold')

    for i, col in enumerate(maternal_cols[:4]):
        ax = axes[i // 2, i % 2]
        df_final[col].dropna().hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
        ax.set_title(col[:60], fontsize=10)
        ax.set_xlabel('Percentage')
        ax.set_ylabel('Number of Districts')
        ax.axvline(df_final[col].mean(), color='red', linestyle='--', label=f'Mean: {df_final[col].mean():.1f}%')
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('01_indicator_distributions.png', dpi=150, bbox_inches='tight')
    print("Saved: 01_indicator_distributions.png")
    plt.close()

    # --- 5b: State-level bar chart ---
    if state_col:
        fig, ax = plt.subplots(figsize=(14, 8))
        indicator = maternal_cols[0]
        state_avg = df_final.groupby(state_col)[indicator].mean().sort_values()
        state_avg.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
        ax.set_xlabel(f'{indicator} (%)')
        ax.set_title(f'State-wise Average: {indicator[:60]}')
        ax.axvline(df_final[indicator].mean(), color='red', linestyle='--', label='National Average')
        ax.legend()
        plt.tight_layout()
        plt.savefig('02_state_comparison.png', dpi=150, bbox_inches='tight')
        print("Saved: 02_state_comparison.png")
        plt.close()

    # --- 5c: Correlation matrix ---
    if len(maternal_cols) >= 3:
        fig, ax = plt.subplots(figsize=(10, 8))
        corr = df_final[maternal_cols].corr()
        im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
        ax.set_xticks(range(len(maternal_cols)))
        ax.set_yticks(range(len(maternal_cols)))
        ax.set_xticklabels([c[:30] for c in maternal_cols], rotation=45, ha='right', fontsize=8)
        ax.set_yticklabels([c[:30] for c in maternal_cols], fontsize=8)
        plt.colorbar(im, ax=ax)
        ax.set_title('Correlation Between Maternal Health Indicators')
        plt.tight_layout()
        plt.savefig('03_correlation_matrix.png', dpi=150, bbox_inches='tight')
        print("Saved: 03_correlation_matrix.png")
        plt.close()

print("\n" + "=" * 70)
print("DONE — Next steps:")
print("  1. Review the outputs and understand which indicators have most variation")
print("  2. Download India district shapefile for choropleth mapping")
print("  3. Register at https://dhsprogram.com for microdata access")
print("  4. Move to 02_regression_analysis.do (Stata) for the econometric analysis")
print("=" * 70)


STEP 5: Visualizations
Saved: 01_indicator_distributions.png
Saved: 02_state_comparison.png
Saved: 03_correlation_matrix.png

DONE — Next steps:
  1. Review the outputs and understand which indicators have most variation
  2. Download India district shapefile for choropleth mapping
  3. Register at https://dhsprogram.com for microdata access
  4. Move to 02_regression_analysis.do (Stata) for the econometric analysis
